# Machine Learning Zoomcamp

## 3. Machine Learning for Classification — Practice

Work through each exercise from memory before running it. If you get stuck, peek at the
course's own solution notebook: `notebook.ipynb` (`../notebook.ipynb`, one level up).

Dataset: [Telco customer churn](https://www.kaggle.com/blastchar/telco-customer-churn)

Plan:

* Data preparation
* Setting up the validation framework
* EDA
* Feature importance: churn rate and risk ratio
* Feature importance: mutual information
* Feature importance: correlation
* One-hot encoding
* Logistic regression
* Training logistic regression with Scikit-Learn
* Model interpretation
* Using the model

In [ ]:
# Import pandas, numpy, matplotlib, seaborn under their usual aliases


## 3.2 Data preparation

1. Download the Telco churn dataset from the URL below and read it into a DataFrame called `df`.
   `data = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-03-churn-prediction/WA_Fn-UseC_-Telco-Customer-Churn.csv'`
2. Normalize the column names: lowercase them and replace spaces with underscores.
3. Find the columns with string (`object`) dtype and normalize their *values* too: lowercase and
   replace spaces with underscores.
4. `totalcharges` looks numeric but reads in as an object column. Coerce it to numeric with
   `pd.to_numeric(..., errors='coerce')` and check how many rows became `NaN`. Fill those with `0`.
5. Convert the `churn` target column from `yes`/`no` strings to `1`/`0` integers.

**Recall:** why does `errors='coerce'` matter here instead of just letting the conversion raise?

In [ ]:
# 1. Download and read the dataset


In [ ]:
# 2. Normalize column names


In [ ]:
# 3. Normalize string values in object columns


In [ ]:
# 4. Coerce totalcharges to numeric, check NaN count, fill with 0


In [ ]:
# 5. Convert churn to 1/0


## 3.3 Setting up the validation framework

1. Use `train_test_split` from `sklearn.model_selection` to split `df` into `df_full_train`
   (80%) and `df_test` (20%), with `random_state=1`.
2. Split `df_full_train` again into `df_train` (75%) and `df_val` (25%) — this yields an overall
   60/20/20 train/val/test split. Use `random_state=1` again.
3. Reset the index on all three DataFrames (`reset_index(drop=True)`).
4. Extract `y_train`, `y_val`, `y_test` as the `.values` of the `churn` column from each split.
5. Delete the `churn` column from `df_train`, `df_val`, `df_test` — why is this step necessary?

**Recall:** why do we split off the test set *before* splitting train/val, rather than doing a
single three-way split in one step?

In [ ]:
# 1-2. Two-step split: full_train/test, then train/val


In [ ]:
# 3. Reset indices


In [ ]:
# 4-5. Extract y arrays, then delete churn from the feature frames


## 3.4 EDA

1. Check `df_full_train.isnull().sum()` for missing values.
2. Look at the distribution of the target: `df_full_train.churn.value_counts(normalize=True)`.
3. Compute the global churn rate as `df_full_train.churn.mean()` and round it to 2 decimals.
4. Split the columns into `numerical` and `categorical` lists by hand (exclude `customerid` and
   `churn` from both).
5. For each categorical column, check `df_full_train[col].nunique()`.

**Recall:** why does the mean of a 0/1 column give you the churn rate directly?

In [ ]:
# 1. Missing value check


In [ ]:
# 2-3. Target distribution and global churn rate


In [ ]:
# 4. Build numerical / categorical column lists


In [ ]:
# 5. Unique value counts per categorical column


## 3.5 Feature importance: churn rate and risk ratio

1. Pick a categorical feature (e.g. `gender`). Group `df_full_train` by it and compute the mean
   churn rate per group with `.groupby('gender').churn.agg(['mean', 'count'])`.
2. For each group, compute the **difference** from the global churn rate (`group_mean - global`).
   A negative difference means that group churns *more* than average.
3. For the same feature, compute the **risk ratio**: `group_mean / global`. A ratio > 1 means the
   group is more likely to churn; < 1 means less likely.
4. Repeat steps 1-3 for `partner` and `contract` — which categories stand out as high risk?

**Recall:** in your own words, what's the difference between what the churn-rate-difference tells
you and what the risk ratio tells you, even though they're built from the same two numbers?

In [ ]:
# 1. Group by gender: mean churn rate and count per group


In [ ]:
# 2. Difference from global churn rate


In [ ]:
# 3. Risk ratio


In [ ]:
# 4. Repeat for partner and contract


## 3.6 Feature importance: mutual information

1. Import `mutual_info_score` from `sklearn.metrics`.
2. Write a `mutual_info_churn_score(series)` function that computes
   `mutual_info_score(series, df_full_train.churn)`.
3. Apply it to every categorical column with `df_full_train[categorical].apply(...)`.
4. Sort the resulting scores descending (`.sort_values(ascending=False)`) — which categorical
   feature carries the most information about churn?

**Recall:** why can mutual information rank categorical features on one common scale, when churn
rate / risk ratio have to be inspected one feature (and one category) at a time?

In [ ]:
# 1-2. Import mutual_info_score, define mutual_info_churn_score(series)


In [ ]:
# 3-4. Apply across categorical columns, sort descending


## 3.7 Feature importance: correlation

1. Compute `df_full_train[numerical].corrwith(df_full_train.churn)` — the correlation of each
   numerical feature with the churn target.
2. For each correlation value, classify it as LOW (`|r| < 0.2`), MEDIUM (`0.2 <= |r| < 0.5`), or
   STRONG (`|r| >= 0.5`).
3. Pick the numerical feature with the strongest correlation and split it into two groups at its
   median. Compare the churn rate of the two groups with `.groupby(...).churn.mean()` to sanity
   check the sign of the correlation.

**Recall:** why does correlation only apply to numerical features, and why did we need mutual
information for the categorical ones instead?

In [ ]:
# 1. Correlation of numerical features with churn


In [ ]:
# 2. Classify each correlation as LOW / MEDIUM / STRONG


In [ ]:
# 3. Median split + churn rate sanity check


## 3.8 One-hot encoding

1. Import `DictVectorizer` from `sklearn.feature_extraction`.
2. Convert `df_train[categorical + numerical]` to a list of dicts with
   `.to_dict(orient='records')`.
3. Fit a `DictVectorizer(sparse=False)` on the training dicts and transform them into `X_train`.
4. Inspect `dv.get_feature_names_out()` — how many columns did the categorical features expand
   into, and what happened to the numerical features?
5. Transform `df_val` the same way into `X_val`, re-using the **already-fitted** `dv` (don't
   re-fit on validation data).

**Recall:** why must `dv` be fit on the training data only, and reused (not refit) for validation
and test?

In [ ]:
# 1-3. DictVectorizer: dicts -> fit_transform on df_train


In [ ]:
# 4. Inspect get_feature_names_out()


In [ ]:
# 5. Transform df_val with the already-fitted dv


## 3.9-3.10 Logistic regression, trained with Scikit-Learn

1. Write a `sigmoid(z)` function: `1 / (1 + np.exp(-z))`. Plot it over `z = np.linspace(-7, 7, 51)`
   to see the S-curve.
2. Import `LogisticRegression` from `sklearn.linear_model`. Train it on `X_train`, `y_train` with
   `solver='liblinear'`, `random_state=1`.
3. Inspect `model.intercept_[0]` and `model.coef_[0]` — how many weights are there, and how does
   that compare to the number of columns in `X_train`?
4. Get hard predictions with `model.predict(X_val)` and soft predictions (probabilities) with
   `model.predict_proba(X_val)` — which column of the soft predictions corresponds to "churn"?
5. Threshold the soft predictions at 0.5 (`y_pred >= 0.5`) and compute accuracy as
   `(y_val == churn_decision).mean()`.

**Recall:** why does logistic regression need the sigmoid on top of the linear part, when plain
linear regression doesn't?

In [ ]:
# 1. sigmoid(z) + plot


In [ ]:
# 2. Train LogisticRegression on X_train, y_train


In [ ]:
# 3. Inspect intercept_ and coef_


In [ ]:
# 4. Hard vs soft predictions on X_val


In [ ]:
# 5. Threshold at 0.5 and compute validation accuracy


## 3.11 Model interpretation

1. `zip` the feature names (`dv.get_feature_names_out()`) with the model's weights
   (`model.coef_[0]`) into a dict, and print it sorted by absolute weight, descending.
2. Pick one categorical feature (e.g. `contract`) and look at only its one-hot weights — do they
   match the risk-ratio direction you found back in 3.5?
3. Train a smaller model using only `['contract', 'tenure', 'totalcharges']` and compare its
   validation accuracy to the full model — how much do we lose by dropping most features?

**Recall:** why does only one of the one-hot columns for a given categorical feature ever
contribute to a specific row's prediction?

In [ ]:
# 1. zip feature names with weights, sort by |weight| descending


In [ ]:
# 2. Inspect one categorical feature's one-hot weights vs. its risk ratios


In [ ]:
# 3. Small model on 3 features, compare validation accuracy


## 3.12 Using the model

1. Combine `df_train` and `df_val` into `df_full_train` (already have this from 3.3, but rebuild
   it explicitly here) and likewise concatenate `y_train`/`y_val` into `y_full_train`.
2. Fit `dv` and the logistic regression model on the combined full-train data.
3. Transform `df_test` with the refit `dv`, predict, and compute test accuracy.
4. Pick a single customer row from `df_test`, convert it to a dict, transform it with `dv`, and get
   the model's churn probability for that one customer. Compare to their actual label.

**Recall:** why is it standard practice to retrain on train+val for the final model, but still
evaluate on a test set that was never touched during any of this?

In [ ]:
# 1. Rebuild df_full_train / y_full_train explicitly


In [ ]:
# 2. Fit dv + model on the combined full-train data


In [ ]:
# 3. Transform df_test, predict, compute test accuracy


In [ ]:
# 4. Predict churn probability for a single test customer


## Wrap-up

In your own words (no code), answer:

1. Walk through why mutual information, risk ratio, and correlation each measure "feature
   importance" differently, and when you'd reach for which one.
2. What does `DictVectorizer` actually do to a row that has both numerical and categorical
   columns, and why do numerical columns pass through unchanged?
3. What's the practical difference between `model.predict(X)` and `model.predict_proba(X)`, and
   why might you want the soft predictions even if you're going to threshold them anyway?